In [19]:
from typing import TypedDict, Annotated, Literal
import sqlite3
import requests

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, SystemMessage,HumanMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_ollama import OllamaEmbeddings
load_dotenv()
import os


In [20]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)
embeddings =OllamaEmbeddings(model="nomic-embed-text", base_url="http://172.31.0.1:11434")

In [21]:
from langgraph.store.memory import InMemoryStore

In [ ]:
store=InMemoryStore()


In [23]:
namespace=("user", "u1")

In [24]:
## adding memories
store.put(namespace, "1", {"data": "user likes waffle"})
store.put(namespace, "2", {"data": "user likes to play football"})

In [25]:
## Retrieve all memories for a specific namepace
store.get(namespace,"1")

Item(namespace=['user', 'u1'], key='1', value={'data': 'user likes waffle'}, created_at='2026-08-24T17:35:37.822188+00:00', updated_at='2026-08-24T17:35:37.822194+00:00')

In [26]:
items=store.search(namespace)
for i in items:
    print(i)

Item(namespace=['user', 'u1'], key='1', value={'data': 'user likes waffle'}, created_at='2026-08-24T17:35:37.822188+00:00', updated_at='2026-08-24T17:35:37.822194+00:00', score=None)
Item(namespace=['user', 'u1'], key='2', value={'data': 'user likes to play football'}, created_at='2026-08-24T17:35:37.822260+00:00', updated_at='2026-08-24T17:35:37.822262+00:00', score=None)


In [27]:
## semantic search
store=InMemoryStore(index={'embed':embeddings,'dims':1536})
namespace2=("user", "u2")
store.put(namespace2, "1", {"data": "user likes pizza"})
store.put(namespace2, "2", {"data": "user likes to watch series"})
store.put(namespace2, "3", {"data": "user is based in india"})
store.put(namespace2, "4", {"data": "user likes to watch cricket also"})
store.put(namespace2, "5", {"data": "user favourite cricket player is virat kohli"})

In [38]:
item2=store.search(namespace2,query="user eats ",limit=2)

In [39]:
for i in item2:
    print(i.value)

{'data': 'user likes pizza'}
{'data': 'user likes to watch cricket also'}
